In [1]:
import requests
import random 
import numpy as np
import tensorflow as tf

In [2]:
book_id = 19033
urls = [
    f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
    f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
]

text = None
for url in urls:
    r = requests.get(url)
    if r.status_code == 200:
        text = r.text

        print("Downloaded: ", url)
        break

# 485 > header
# 485:54815
# 54883 total

Downloaded:  https://www.gutenberg.org/files/19033/19033-0.txt


In [3]:
# chopped header and footer and lowered everything
norm_text = text[485:54815].lower()
uniq_chars = sorted(set(norm_text))
print(f"Printing {len(uniq_chars)} characters: {uniq_chars}")

Printing 45 characters: ['\n', '\r', ' ', '!', '"', "'", '(', ')', '*', ',', '-', '.', ':', ';', '?', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'ù']


In [10]:
# global dictionaries
char2int = {}
int2char = {}

def char_to_int(chars):
    char2int.clear()          
    for i, ch in enumerate(chars):
        char2int[ch] = i
    print(char2int)
    return char2int

def int_to_char(mapping):
    int2char.clear()
    for ch, idx in mapping.items():
        int2char[idx] = ch
    print(int2char)
    return int2char

In [11]:
char2int = char_to_int(uniq_chars)
int2char = int_to_char(char2int)

{'\n': 0, '\r': 1, ' ': 2, '!': 3, '"': 4, "'": 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, ':': 12, ';': 13, '?': 14, '[': 15, ']': 16, '_': 17, 'a': 18, 'b': 19, 'c': 20, 'd': 21, 'e': 22, 'f': 23, 'g': 24, 'h': 25, 'i': 26, 'j': 27, 'k': 28, 'l': 29, 'm': 30, 'n': 31, 'o': 32, 'p': 33, 'q': 34, 'r': 35, 's': 36, 't': 37, 'u': 38, 'v': 39, 'w': 40, 'x': 41, 'y': 42, 'z': 43, 'ù': 44}
{0: '\n', 1: '\r', 2: ' ', 3: '!', 4: '"', 5: "'", 6: '(', 7: ')', 8: '*', 9: ',', 10: '-', 11: '.', 12: ':', 13: ';', 14: '?', 15: '[', 16: ']', 17: '_', 18: 'a', 19: 'b', 20: 'c', 21: 'd', 22: 'e', 23: 'f', 24: 'g', 25: 'h', 26: 'i', 27: 'j', 28: 'k', 29: 'l', 30: 'm', 31: 'n', 32: 'o', 33: 'p', 34: 'q', 35: 'r', 36: 's', 37: 't', 38: 'u', 39: 'v', 40: 'w', 41: 'x', 42: 'y', 43: 'z', 44: 'ù'}


In [21]:
def create_sequence(norm_text, char2int, seq_length, step=1):
    X = []
    y = []
    for i in range(0, len(norm_text) - seq_length, step):
        seq_in = norm_text[i : i + seq_length]
        seq_out = norm_text[i + seq_length]

        X.append([char2int[char] for char in seq_in])
        y.append(char2int[seq_out])

    return X, y

In [24]:
seq_length = 60
X, y = create_sequence(norm_text, char2int, seq_length, step=3)
X = np.array(X)
y = np.array(y)

print("training samples ", len(X))
print("First X ", X[0])
print("First y ", y[0])

training samples  18090
First X  [ 1  0  1  0 18 29 26 20 22  5 36  2 18 21 39 22 31 37 38 35 22 36  2 26
 31  2 40 32 31 21 22 35 29 18 31 21  1  0  1  0 15 26 29 29 38 36 37 35
 18 37 26 32 31 16  1  0  1  0  1  0]
First y  1


In [27]:
V = len(uniq_chars)
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(
        input_dim=V, 
        output_dim=8,
        input_length=60),
    tf.keras.layers.LSTM(256),
    tf.keras.layers.Dense(V, activation='softmax')
])

In [28]:
model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

In [ ]:

training = model.fit(X, y, epochs=20, batch_size=128)

Epoch 1/20


In [ ]:
seed = norm_text[50:112]
upd_seed = create_sequence(seed, char2int, seq_length)

for x in range(15):
    model.predict(upd_seed.reshape(1, seq_length))

ValueError: Unrecognized data type: x=([[26, 32, 31, 16, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 26, 10, 10, 21, 32, 40, 31, 2, 37, 25, 22, 2, 35, 18, 19, 19, 26, 37, 10, 25, 32, 29, 22, 1, 0, 1, 0, 1, 0, 18, 29, 26, 20, 22, 2, 40, 18, 36, 2, 19, 22, 24, 26, 31, 31, 26], [32, 31, 16, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 26, 10, 10, 21, 32, 40, 31, 2, 37, 25, 22, 2, 35, 18, 19, 19, 26, 37, 10, 25, 32, 29, 22, 1, 0, 1, 0, 1, 0, 18, 29, 26, 20, 22, 2, 40, 18, 36, 2, 19, 22, 24, 26, 31, 31, 26, 31]], [31, 24]) (of type <class 'tuple'>)